# ⚙️ 02 — Preprocesamiento
**Proyecto:** Detección de Fraude Financiero con ML + LLM Explicativo  

---
**Objetivo:** Aplicar el pipeline completo: imputación, feature engineering, escalado, SMOTE y división train/val/test.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, os, joblib
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
TARGET = 'Class'
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)
print('✅ Imports OK')

## 1. Cargar dataset del EDA

In [ ]:
if os.path.exists('../data/processed/dataset_eda.parquet'):
    df = pd.read_parquet('../data/processed/dataset_eda.parquet')
else:
    # Regenerar sintético si no existe
    from sklearn.datasets import make_classification
    X_syn, y_syn = make_classification(
        n_samples=10_000, n_features=30, n_informative=15, n_redundant=5,
        weights=[0.9965, 0.0035], flip_y=0, random_state=RANDOM_STATE
    )
    feature_names = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount']
    df = pd.DataFrame(X_syn, columns=feature_names)
    df['Time']   = np.abs(df['Time']) * 3600
    df['Amount'] = np.abs(df['Amount']) * 100 + 10
    df['Class']  = y_syn
    for col in ['V1', 'V3', 'Amount']:
        df.loc[df.sample(frac=0.04, random_state=RANDOM_STATE).index, col] = np.nan
    print('⚠️  Usando dataset sintético. Ejecuta primero 01_eda.ipynb.')

print(f'✅ Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
display(df.head(3))

## 2. Feature Engineering

In [ ]:
# Hora del día (0-23) desde el timestamp
df['hour_of_day'] = (df['Time'] % 86400) / 3600

# Log-transform del monto (reduce efecto de outliers)
df['log_amount'] = np.log1p(df['Amount'].fillna(df['Amount'].median()))

# Eliminar columnas originales reemplazadas
df = df.drop(columns=['Time', 'Amount'])

print('✅ Feature engineering aplicado:')
print('   + hour_of_day  (Time % 86400 / 3600)')
print('   + log_amount   (log1p(Amount))')
print('   - Time, Amount eliminados')
print(f'\nDataset final: {df.shape[0]:,} filas × {df.shape[1]} columnas')

## 3. Separar features y target

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Features numéricas ({len(num_cols)}): {num_cols}')
print(f'Features categóricas ({len(cat_cols)}): {cat_cols}')
print(f'Variable objetivo: {TARGET}  |  Clases: {sorted(y.unique())}')

## 4. División Train / Val / Test  (70 / 15 / 15)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

for nombre, Xs, ys in [('Train', X_train, y_train), ('Val', X_val, y_val), ('Test', X_test, y_test)]:
    pct_fraude = ys.mean() * 100
    print(f'{nombre:6s}: {len(Xs):>7,} muestras  |  fraude: {pct_fraude:.3f}%')

## 5. Pipeline sklearn

In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[('num', num_pipeline, num_cols)],
    remainder='drop'
)

# Fit SOLO en train — transform en val y test
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
X_test_proc  = preprocessor.transform(X_test)

print(f'Shape train procesado:  {X_train_proc.shape}')
print(f'Shape val procesado:    {X_val_proc.shape}')
print(f'Shape test procesado:   {X_test_proc.shape}')

# Verificar que no quedaron nulos
for nombre, arr in [('Train', X_train_proc), ('Val', X_val_proc), ('Test', X_test_proc)]:
    n = np.isnan(arr).sum()
    print(f'  Nulos en {nombre}: {"✅ 0" if n == 0 else f"⚠️ {n}"}')

## 6. SMOTE — balanceo sintético en train

In [ ]:
try:
    from imblearn.over_sampling import SMOTE
    sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
    X_train_bal, y_train_bal = sm.fit_resample(X_train_proc, y_train)
    print(f'✅ SMOTE aplicado:')
    print(f'   Antes:  {X_train_proc.shape[0]:,} muestras  |  fraude: {y_train.mean()*100:.3f}%')
    print(f'   Después:{X_train_bal.shape[0]:,} muestras  |  fraude: {y_train_bal.mean()*100:.1f}%')
except ImportError:
    print('⚠️  imbalanced-learn no instalado. Usando datos sin balancear.')
    print('   Instalar: pip install imbalanced-learn')
    X_train_bal, y_train_bal = X_train_proc, y_train.values

# Convertir a arrays numpy para consistencia
y_train_bal = np.array(y_train_bal)
y_val_arr   = y_val.values
y_test_arr  = y_test.values

## 7. Guardar artefactos

In [ ]:
np.save('../data/processed/X_train.npy', X_train_bal)
np.save('../data/processed/X_val.npy',   X_val_proc)
np.save('../data/processed/X_test.npy',  X_test_proc)
np.save('../data/processed/y_train.npy', y_train_bal)
np.save('../data/processed/y_val.npy',   y_val_arr)
np.save('../data/processed/y_test.npy',  y_test_arr)

joblib.dump(preprocessor, '../models/preprocessor.joblib')

# Guardar también nombres de features para SHAP
pd.Series(num_cols).to_csv('../data/processed/feature_names.csv', index=False)

print('✅ Artefactos guardados en data/processed/ y models/')
print('\n→ Continuar con: 03_modeling.ipynb')